# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research question

**Can observable March 2026 search and engagement signals help prioritize content observations for human review when the next-month outcome is a measured decline in clicks?**

### Decision supported

The output is a ranked review queue. A content or SEO reviewer can use it to decide which observations deserve investigation first and which should remain in monitoring.

The unit of analysis is a **client-content observation aggregated across March 2026**. The model target is the observed April click-decline outcome for observations that had March click demand.

The decision-support goal is prioritization, not automatic publishing or content modification.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The analysis uses the FlyRank internship warehouse release available through the assignment's gated dataset.

### Windows

- **March 2026:** features
- **April 2026:** next-month outcome

### March features

The analysis aggregates:

- Google Search Console impressions
- Google Search Console clicks
- average position
- GA4 sessions
- GA4 engaged sessions
- organic sessions
- AI-referral sessions
- scroll events
- derived CTR
- derived engagement rate

### April outcome

April impressions, clicks, sessions, and engaged sessions are aggregated to define the next-month outcome.

### Exclusions

The model does not use client IDs or content IDs as predictive features. The client identifier is used only for grouped validation.

The target is not used as a feature. Future April information is not used to construct March features.

The notebook does not expose client names, domains, private queries, credentials, or raw production exports.

**Public-safe framing:** identifiers remain pseudonymous and are used only to establish the grouping boundary required for honest validation.


In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    accuracy_score,
)

print("Libraries loaded.")


Libraries loaded.


In [2]:
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN is not available in Colab secrets.")

login(token=HF_TOKEN)
print("Hugging Face login complete.")


Hugging Face login complete.


In [3]:
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN,
)

april_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN,
)

print("March file ready.")
print("April file ready.")


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March file ready.
April file ready.


In [4]:
feature_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_ai",
    "scroll_events",
]

march_raw = pd.read_parquet(march_file, columns=feature_columns)

print("March rows:", len(march_raw))
print("March date range:", march_raw["report_date"].min(), "to", march_raw["report_date"].max())


March rows: 9841378
March date range: 2026-03-01 to 2026-03-31


In [5]:
march_features = (
    march_raw
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_avg_position=("gsc_avg_position", "mean"),
        march_sessions=("ga4_sessions", "sum"),
        march_engaged_sessions=("ga4_engaged_sessions", "sum"),
        march_organic_sessions=("sessions_organic", "sum"),
        march_ai_sessions=("sessions_ai", "sum"),
        march_scroll_events=("scroll_events", "sum"),
    )
)

march_features["march_ctr"] = np.where(
    march_features["march_impressions"] > 0,
    march_features["march_clicks"] / march_features["march_impressions"],
    np.nan,
)

march_features["march_engagement_rate"] = np.where(
    march_features["march_sessions"] > 0,
    march_features["march_engaged_sessions"] / march_features["march_sessions"],
    np.nan,
)

print("Aggregated March rows:", len(march_features))


Aggregated March rows: 331437


In [6]:
april_raw = pd.read_parquet(
    april_file,
    columns=[
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ga4_sessions",
        "ga4_engaged_sessions",
    ],
)

april_outcome = (
    april_raw
    .groupby(["client_hash_id", "content_hash_id"], as_index=False)
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum"),
        april_sessions=("ga4_sessions", "sum"),
        april_engaged_sessions=("ga4_engaged_sessions", "sum"),
    )
)

print("April rows:", len(april_raw))
print("Aggregated April rows:", len(april_outcome))


April rows: 10424730
Aggregated April rows: 362172


In [7]:
model_df = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

model_df["future_decline"] = np.where(
    model_df["march_clicks"] > 0,
    (model_df["april_clicks"] < model_df["march_clicks"]).astype(int),
    np.nan,
)

model_df = model_df.dropna(subset=["future_decline"]).copy()
model_df["future_decline"] = model_df["future_decline"].astype(int)

print("Modeling rows:", len(model_df))
print("Observed future decline rate:", round(model_df["future_decline"].mean(), 4))


Modeling rows: 68837
Observed future decline rate: 0.6552


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Target

An observation is labeled `future_decline = 1` when its April clicks are lower than its March clicks, provided March clicks are greater than zero.

This is an observed next-month outcome definition. It is not a label for "content quality" or "refresh success."

### Features

The model uses the same ten March features as the validated Week-5 model:

1. March impressions
2. March clicks
3. March average position
4. March sessions
5. March engaged sessions
6. March organic sessions
7. March AI sessions
8. March scroll events
9. March CTR
10. March engagement rate

### Model

A Logistic Regression pipeline is used with:

- median imputation;
- standardization;
- Logistic Regression with `max_iter=1000`;
- `random_state=42`.

### Validation

The Week-5 grouped split is reproduced with `GroupShuffleSplit`, 20% test size, and `random_state=42`.

The grouping variable is the pseudonymous client ID. Therefore observations from the same client are kept on the same side of the split.

### Baseline

The baseline always predicts the training-set majority-class probability. This is deliberately transparent and is evaluated on the **same test split** as the model.

### Leakage checks

- April outcome fields are not model features.
- `future_decline` is not a model feature.
- Client and content IDs are not predictive features.
- All model features are March-only.
- The test set is separated by client group.


In [8]:
feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_sessions",
    "march_engaged_sessions",
    "march_organic_sessions",
    "march_ai_sessions",
    "march_scroll_events",
    "march_ctr",
    "march_engagement_rate",
]

X = model_df[feature_cols].copy()
y = model_df["future_decline"].copy()
groups = model_df["client_hash_id"].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

test_rows = model_df.iloc[test_idx].copy()

print("Feature matrix:", X.shape)
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())
print("Train decline rate:", round(y_train.mean(), 4))
print("Test decline rate:", round(y_test.mean(), 4))


Feature matrix: (68837, 10)
Train rows: 63775
Test rows: 5062
Train clients: 35
Test clients: 9
Train decline rate: 0.653
Test decline rate: 0.6831


In [9]:
model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

model_probability = model.predict_proba(X_test)[:, 1]
model_prediction = (model_probability >= 0.50).astype(int)

baseline_probability_value = float(y_train.mean())
baseline_probability = np.full(len(y_test), baseline_probability_value)
baseline_prediction = np.full(len(y_test), int(round(baseline_probability_value)))

print("Week-5 model reproduced.")
print("Baseline probability:", round(baseline_probability_value, 4))


Week-5 model reproduced.
Baseline probability: 0.653


In [10]:
results = pd.DataFrame([
    {
        "method": "majority_probability_baseline",
        "roc_auc": roc_auc_score(y_test, baseline_probability),
        "average_precision": average_precision_score(y_test, baseline_probability),
        "brier_score": brier_score_loss(y_test, baseline_probability),
        "accuracy_at_0_50": accuracy_score(y_test, baseline_prediction),
    },
    {
        "method": "week_5_logistic_regression",
        "roc_auc": roc_auc_score(y_test, model_probability),
        "average_precision": average_precision_score(y_test, model_probability),
        "brier_score": brier_score_loss(y_test, model_probability),
        "accuracy_at_0_50": accuracy_score(y_test, model_prediction),
    },
])

display(results.round(4))


,method,roc_auc,average_precision,brier_score,accuracy_at_0_50
0,majority_probability_baseline,0.5000,0.6831,0.2174,0.6831
1,week_5_logistic_regression,0.6132,0.7813,0.2110,0.6839


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The table above evaluates both approaches on the same grouped test split.

The majority baseline is intentionally simple. Because the test-set class balance is already high, accuracy alone is not a strong discrimination measure. ROC AUC and average precision provide more useful comparisons of ranking/discrimination, while Brier score describes probability quality.

The model should only be described as useful where its measured test-set performance is better or otherwise informative relative to this baseline. The result does not establish causality or future refresh impact.


In [11]:
# Model interpretation: standardized Logistic Regression coefficients.
classifier = model.named_steps["classifier"]
coefficients = pd.Series(
    classifier.coef_[0],
    index=feature_cols,
).sort_values(key=np.abs, ascending=False)

coefficient_table = coefficients.rename("standardized_coefficient").reset_index()
coefficient_table = coefficient_table.rename(columns={"index": "feature"})

display(coefficient_table.round(4))


,feature,standardized_coefficient
0,march_ctr,0.7566
1,march_sessions,-0.1242
2,march_scroll_events,0.0938
3,march_organic_sessions,-0.0603
4,march_impressions,0.0580
5,march_engaged_sessions,0.0553
6,march_engagement_rate,-0.0426
7,march_ai_sessions,-0.0223
8,march_clicks,-0.0175
9,march_avg_position,0.0094


### Ranked review queue

A model probability by itself is **not** treated as an action priority.

The previous Week-5-derived queue exposed an important weakness: observations with only one impression and one click could receive probabilities near 1.0 and appear at the top of the queue. That is not enough evidence to justify immediate content work.

Therefore this capstone adds a separate evidence-aware prioritization layer:

- model probability measures the model signal;
- evidence strength reflects observable exposure;
- the recommended action still depends on the observable archetype;
- low-evidence observations remain monitoring candidates.

This is a decision-policy layer, not a claim that the model probability is wrong.


In [12]:
queue = test_rows[
    [
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "march_clicks",
        "march_ctr",
        "march_avg_position",
        "march_sessions",
        "march_engagement_rate",
        "march_organic_sessions",
        "march_ai_sessions",
        "march_scroll_events",
    ]
].copy()

queue["model_probability"] = model_probability

queue["reason_code"] = "monitor"

mask = (
    (queue["march_impressions"] >= 500)
    & (queue["march_ctr"] < 0.005)
)
queue.loc[mask, "reason_code"] = "low_ctr_visible_page"

mask = (
    (queue["reason_code"] == "monitor")
    & (queue["march_sessions"] >= 30)
    & (queue["march_engagement_rate"].notna())
    & (queue["march_engagement_rate"] < 0.30)
)
queue.loc[mask, "reason_code"] = "low_engagement_visible_page"

mask = (
    (queue["reason_code"] == "monitor")
    & (queue["march_impressions"] >= 100)
    & (queue["march_clicks"] > 0)
)
queue.loc[mask, "reason_code"] = "demand_present"

mask = (
    (queue["reason_code"] == "monitor")
    & (queue["march_impressions"] >= 500)
)
queue.loc[mask, "reason_code"] = "visible_page"

queue["archetype"] = "low_evidence"

queue.loc[
    (queue["march_impressions"] >= 500)
    & (queue["march_ctr"] < 0.005),
    "archetype",
] = "high_visibility_low_ctr"

queue.loc[
    (queue["march_sessions"] >= 30)
    & (queue["march_engagement_rate"].notna())
    & (queue["march_engagement_rate"] < 0.30),
    "archetype",
] = "high_traffic_low_engagement"

queue.loc[
    (queue["march_impressions"] >= 100)
    & (queue["march_clicks"] > 0)
    & (queue["archetype"] == "low_evidence"),
    "archetype",
] = "demand_with_clicks"

queue.loc[
    (queue["march_impressions"] >= 500)
    & (queue["archetype"] == "low_evidence"),
    "archetype",
] = "visible_without_strong_issue"

# Evidence factor: low-exposure observations receive less action priority.
# It is bounded between 0 and 1 and uses impressions only as an evidence-strength signal.
evidence_factor = np.log1p(queue["march_impressions"]) / np.log1p(5000)
queue["evidence_factor"] = evidence_factor.clip(0, 1)

queue["priority_score"] = (
    queue["model_probability"]
    * queue["evidence_factor"]
    * 100
)

action_map = {
    "high_visibility_low_ctr": "review_title_snippet_intent",
    "high_traffic_low_engagement": "review_content_experience",
    "demand_with_clicks": "review_refresh_opportunity",
    "visible_without_strong_issue": "human_review",
    "low_evidence": "monitor",
}

queue["recommended_action"] = (
    queue["archetype"]
    .map(action_map)
    .fillna("human_review")
)

queue["human_review_required"] = True
queue["automation_allowed"] = False

queue = queue.sort_values(
    by=["priority_score", "march_impressions"],
    ascending=[False, False],
).reset_index(drop=True)

queue["rank"] = queue.index + 1

display(
    queue[
        [
            "rank",
            "model_probability",
            "evidence_factor",
            "priority_score",
            "archetype",
            "reason_code",
            "recommended_action",
        ]
    ].head(20)
)


,rank,model_probability,evidence_factor,priority_score,archetype,reason_code,recommended_action
0,1,0.932841,0.843880,78.720572,high_traffic_low_engagement,low_engagement_visible_page,review_content_experience
1,2,0.759747,1.000000,75.974687,high_visibility_low_ctr,low_ctr_visible_page,review_title_snippet_intent
2,3,0.727738,1.000000,72.773816,high_traffic_low_engagement,low_engagement_visible_page,review_content_experience
3,4,0.710704,1.000000,71.070441,high_traffic_low_engagement,low_engagement_visible_page,review_content_experience
4,5,0.680070,1.000000,68.006976,high_traffic_low_engagement,low_engagement_visible_page,review_content_experience
5,6,0.667809,1.000000,66.780897,high_visibility_low_ctr,low_ctr_visible_page,review_title_snippet_intent
6,7,0.669941,0.992412,66.485743,high_traffic_low_engagement,low_engagement_visible_page,review_content_experience
7,8,0.663819,1.000000,66.381857,high_traffic_low_engagement,low_engagement_visible_page,review_content_experience
8,9,0.660654,1.000000,66.065396,high_visibility_low_ctr,low_ctr_visible_page,review_title_snippet_intent
9,10,0.660522,1.000000,66.052207,high_visibility_low_ctr,low_ctr_visible_page,review_title_snippet_intent


In [13]:
print("Top 20 ranked observations:")
display(
    queue.head(20)[
        [
            "rank",
            "model_probability",
            "march_impressions",
            "march_clicks",
            "march_ctr",
            "archetype",
            "recommended_action",
        ]
    ]
)

print("\nArchetype counts:")
display(queue["archetype"].value_counts().rename_axis("archetype").reset_index(name="observations"))

print("\nLow-evidence observations in top 20:",
      int((queue.head(20)["archetype"] == "low_evidence").sum()))


Top 20 ranked observations:


,rank,model_probability,march_impressions,march_clicks,march_ctr,archetype,recommended_action
0,1,0.932841,1322,24,0.018154,high_traffic_low_engagement,review_content_experience
1,2,0.759747,89332,4,0.000045,high_visibility_low_ctr,review_title_snippet_intent
2,3,0.727738,14738,315,0.021373,high_traffic_low_engagement,review_content_experience
3,4,0.710704,19657,199,0.010124,high_traffic_low_engagement,review_content_experience
4,5,0.680070,9630,92,0.009553,high_traffic_low_engagement,review_content_experience
5,6,0.667809,28795,37,0.001285,high_visibility_low_ctr,review_title_snippet_intent
6,7,0.669941,4687,64,0.013655,high_traffic_low_engagement,review_content_experience
7,8,0.663819,7238,111,0.015336,high_traffic_low_engagement,review_content_experience
8,9,0.660654,22327,20,0.000896,high_visibility_low_ctr,review_title_snippet_intent
9,10,0.660522,16519,19,0.001150,high_visibility_low_ctr,review_title_snippet_intent



Archetype counts:


,archetype,observations
0,demand_with_clicks,2561
1,high_visibility_low_ctr,1473
2,low_evidence,886
3,high_traffic_low_engagement,142



Low-evidence observations in top 20: 0


## 5. Limitations

*What this work cannot claim.*

This work cannot claim:

- that a content refresh caused a performance change;
- that a refresh will reverse an observed decline;
- that the model predicts Google's ranking algorithm;
- that a high model probability proves a page has a content problem;
- that the model captures editorial quality, search intent, technical SEO, business value, or every reason a page changes;
- that the model will remain calibrated on future data without monitoring;
- that sparse observations provide strong evidence for an action.

The April decline label is a retrospective measurement used for model evaluation. It is not a direct measure of refresh success.

The ranked queue is therefore **directional decision-support** and requires human review.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The queue maps observable patterns to review actions:

1. **High visibility + low CTR** → review title, snippet, and search-intent alignment.
2. **High traffic + low engagement** → review content experience and intent match.
3. **Demand with clicks** → review for a possible refresh opportunity.
4. **Visible without a strong issue** → human review.
5. **Low evidence** → monitor rather than automatically escalating.

These are recommendations for investigation, not automatic content instructions.

The model score and evidence-aware priority score should be considered together with the reason code and archetype.


In [14]:
monitoring_snapshot = pd.DataFrame(
    {
        "metric": [
            "queue_rows",
            "mean_model_probability",
            "high_priority_share",
            "low_evidence_share",
            "human_review_required",
            "automatic_actions_allowed",
        ],
        "value": [
            len(queue),
            queue["model_probability"].mean(),
            (queue["model_probability"] >= 0.70).mean(),
            (queue["archetype"] == "low_evidence").mean(),
            queue["human_review_required"].all(),
            queue["automation_allowed"].any(),
        ],
    }
)

display(monitoring_snapshot.round(4))

archetype_monitoring = (
    queue
    .groupby("archetype", as_index=False)
    .agg(
        observations=("content_hash_id", "count"),
        mean_probability=("model_probability", "mean"),
        mean_priority_score=("priority_score", "mean"),
        mean_impressions=("march_impressions", "mean"),
        mean_clicks=("march_clicks", "mean"),
    )
    .sort_values("mean_probability", ascending=False)
)

display(archetype_monitoring.round(4))


,metric,value
0,queue_rows,5062
1,mean_model_probability,0.668681
2,high_priority_share,0.120308
3,low_evidence_share,0.17503
4,human_review_required,True
5,automatic_actions_allowed,False


,archetype,observations,mean_probability,mean_priority_score,mean_impressions,mean_clicks
3,low_evidence,886,0.7600,29.5838,44.6445,1.2460
0,demand_with_clicks,2561,0.6554,45.4790,541.7314,4.5748
1,high_traffic_low_engagement,142,0.6460,59.0513,5453.0282,66.1831
2,high_visibility_low_ctr,1473,0.6390,54.3574,2040.9925,4.3618


In [15]:
decay_analysis = model_df[
    [
        "march_impressions",
        "march_clicks",
        "march_ctr",
        "march_sessions",
        "march_engagement_rate",
        "future_decline",
    ]
].copy()

decay_analysis["visibility_band"] = pd.cut(
    decay_analysis["march_impressions"],
    bins=[-1, 99, 499, np.inf],
    labels=[
        "low_visibility",
        "medium_visibility",
        "high_visibility",
    ],
)

decay_summary = (
    decay_analysis
    .groupby("visibility_band", observed=True)
    .agg(
        observations=("future_decline", "size"),
        observed_decline_rate=("future_decline", "mean"),
    )
    .reset_index()
)

display(decay_summary.round(4))


,visibility_band,observations,observed_decline_rate
0,low_visibility,5175,0.8319
1,medium_visibility,12534,0.6903
2,high_visibility,51128,0.6287


### Retrospective decay check

The visibility-band table is descriptive. It shows how the measured April click-decline rate varied across March visibility bands.

It should be interpreted as an observed directional pattern. It does not show that visibility causes decline or that refreshing a page would reverse the outcome.


In [16]:
# Create paper-ready artifacts.
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

results_path = output_dir / "capstone_model_vs_baseline.csv"
queue_path = output_dir / "w08_content_action_queue.csv"
archetype_path = output_dir / "w08_archetype_action_summary.csv"
monitoring_path = output_dir / "w08_monitoring_snapshot.csv"
decay_path = output_dir / "w08_decay_refresh_summary.csv"

results.to_csv(results_path, index=False)

queue.to_csv(queue_path, index=False)

archetype_summary = (
    queue
    .groupby(
        ["archetype", "recommended_action", "reason_code"],
        as_index=False,
    )
    .agg(
        observations=("content_hash_id", "count"),
        mean_probability=("model_probability", "mean"),
        mean_priority_score=("priority_score", "mean"),
    )
    .sort_values("mean_probability", ascending=False)
)

archetype_summary.to_csv(archetype_path, index=False)
monitoring_snapshot.to_csv(monitoring_path, index=False)
decay_summary.to_csv(decay_path, index=False)

print("Saved:")
for path in [
    results_path,
    queue_path,
    archetype_path,
    monitoring_path,
    decay_path,
]:
    print(" -", path)


Saved:
 - work/outputs/capstone_model_vs_baseline.csv
 - work/outputs/w08_content_action_queue.csv
 - work/outputs/w08_archetype_action_summary.csv
 - work/outputs/w08_monitoring_snapshot.csv
 - work/outputs/w08_decay_refresh_summary.csv


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The research paper should reuse measured outputs from this notebook rather than manually retyping numbers.

Recommended paper artifacts:

- model-vs-baseline metrics table from `capstone_model_vs_baseline.csv`;
- ranked recommendation summary from `w08_archetype_action_summary.csv`;
- monitoring snapshot from `w08_monitoring_snapshot.csv`;
- retrospective visibility/decline table from `w08_decay_refresh_summary.csv`.

A chart can be generated from these outputs in the paper/deployment workflow. The notebook remains the reproducible source of truth.

### Paper-facing interpretation

The central result should be written using the actual fresh run of this notebook. Do not copy old numbers if a fresh run produces different values.

Use language such as:

- **measured**
- **observed**
- **directional**
- **associated with**
- **decision-support**

Avoid:

- caused
- guaranteed
- proves
- predicts Google's algorithm
- will improve rankings


## ML-12 — 5-minute demo outline

**0:00–0:45 — Question**

What decision are we supporting, and why is next-month click decline a useful retrospective outcome?

**0:45–1:45 — Data + method**

Show the March feature window, April outcome, ten March features, grouped-by-client validation, and leakage controls.

**1:45–2:45 — One chart / result**

Show the model-vs-baseline evaluation from the fresh run. Explain the strongest honest result without overstating it.

**2:45–3:45 — Recommendation engine**

Show the ranked queue and explain the difference between model probability and evidence-aware action priority.

**3:45–4:30 — Important correction**

Show why a one-impression/one-click observation with a near-1.0 model probability should not automatically become the top content action.

**4:30–5:00 — Recommendation + limits**

Explain one recommended human review action and finish with the key limitation: this is directional decision-support, not causal proof or automatic publishing.


## ML-12 — Shareable social post

I built a reproducible content-opportunity scoring workflow on real search-performance data. The work uses March search and engagement signals, a grouped Logistic Regression model, and a retrospective April click-decline outcome to rank observations for human review. The key engineering lesson was separating model probability from action priority: sparse observations can receive extreme probabilities without providing enough evidence for an immediate content decision. The final system is therefore designed as human-reviewed, evidence-aware decision support rather than automatic content optimization.


## ML-12 — Employer-facing summary

I built a reproducible ML decision-support workflow for content opportunity scoring using real search-performance data. I aggregated March features, defined a transparent next-month click-decline outcome, validated a Logistic Regression model with client-grouped splitting and leakage controls, and compared it with a simple majority baseline on the same test split. The result is a human-reviewed ranked action queue that combines model signal, observable evidence, reason codes, archetypes, monitoring, and honest limitations instead of automatically changing content.


## Self-check

Before submission, confirm:

- [x] Every section is filled with markdown reasoning and supporting code.
- [x] The notebook runs top-to-bottom without errors.
- [x] The model and baseline are evaluated on the same grouped test split.
- [x] The base rate is reported.
- [x] Leakage checks are documented.
- [x] Client/content identifiers are not used as predictive features.
- [x] No client names, domains, private queries, credentials, or raw exports appear.
- [x] Claims use careful language: observed, measured, directional, decision-support.
- [x] The low-evidence ranking problem is handled explicitly.
- [x] Human review is required.
- [x] Automatic publishing/content changes are not enabled.
- [x] Paper-ready outputs are written to `work/outputs/`.
- [x] ML-12 demo outline is present.
- [x] ML-12 social post is present.
- [x] ML-12 employer-facing summary is present.
- [x] The notebook is committed under `work/notebooks/capstone.ipynb`.
